In [281]:
!rm -rf sample_data/

In [282]:
# ---- Imports ----
import os
import zipfile
import requests
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from tqdm import tqdm

In [283]:

def download_file_if_not_exists(url, filename):
    """Downloads a file from a URL only if it doesn't already exist locally.

    Args:
        url: The URL of the file.
        filename: The name of the file to save to.
    """
    if os.path.exists(filename):
        print(f"File '{filename}' already exists locally. Skipping download.")
        return

    try:
        response = requests.get(url)

        if response.status_code == 200:
            with requests.get(url, stream=True) as r:
                r.raise_for_status()
                with open(filename, 'wb') as f:
                    for chunk in r.iter_content(chunk_size=8192):
                        f.write(chunk)
            print(f"Downloaded file from {url} to {filename}")
        else:
            print(f"File not found at {url}")

    except requests.exceptions.RequestException as e:
        print(f"Error downloading file: {e}")

# Define URLs
url1 = 'https://github.com/JeremyTubongbanua/sofe4620u-nba/raw/trunk/basketball_player_removal_mlp_model_3.pt'
url2 = 'https://github.com/JeremyTubongbanua/sofe4620u-nba/raw/trunk/data.zip'

filename1 = os.path.basename(url1)
filename2 = os.path.basename(url2)

download_file_if_not_exists(url1, filename1)
download_file_if_not_exists(url2, filename2)

File 'basketball_player_removal_mlp_model_3.pt' already exists locally. Skipping download.
File 'data.zip' already exists locally. Skipping download.


In [284]:
with zipfile.ZipFile('data.zip', 'r') as zip_ref:
    zip_ref.extractall('.')

In [285]:
# ---- Read data files -----
with open("data/games.txt", "r") as f:
    games = [line.strip() for line in f.readlines()]

with open("data/names.txt", "r") as f:
    names = [line.strip() for line in f.readlines()]

with open("data/seasons.txt", "r") as f:
    seasons = [int(line.strip()) for line in f.readlines()]

with open("data/teams.txt", "r") as f:
    teams = [line.strip() for line in f.readlines()]

names_dict = {name: i for i, name in enumerate(names)}
games_dict = {game: i for i, game in enumerate(games)}
teams_dict = {team: i for i, team in enumerate(teams)}
seasons_dict = {season: i for i, season in enumerate(seasons)}

In [286]:
nba_test_df = pd.read_csv('./data/NBA_test.csv')
nba_test_df.head()

,season,home_team,away_team,starting_min,home_0,home_1,home_2,home_3,home_4,away_0,away_1,away_2,away_3,away_4
0,2007,IND,BOS,18,Danny Granger,Darrell Armstrong,Keith McLeod,Mike Dunleavy,?,Allan Ray,Gerald Green,Kendrick Perkins,Ryan Gomes,Sebastian Telfair
1,2007,HOU,DAL,16,Bonzi Wells,?,Juwan Howard,Luther Head,Tracy McGrady,Austin Croshere,Erick Dampier,Greg Buckner,Jason Terry,Josh Howard
2,2007,SAS,POR,39,Beno Udrih,?,Bruce Bowen,Matt Bonner,Tim Duncan,Brandon Roy,Jamaal Magloire,Jarrett Jack,Juan Dixon,Zach Randolph
3,2007,MIN,BOS,21,?,Kevin Garnett,Randy Foye,Ricky Davis,Trenton Hassell,Al Jefferson,Brian Scalabrine,Delonte West,Paul Pierce,Ryan Gomes
4,2007,MEM,LAL,19,Chucky Atkins,Hakim Warrick,Mike Miller,Rudy Gay,?,Andrew Bynum,Kobe Bryant,Lamar Odom,Luke Walton,Smush Parker


In [287]:
nba_test_labels_df = pd.read_csv('./data/NBA_test_labels.csv')
nba_test_labels_df.head()

,removed_value
0,Troy Murphy
1,Chuck Hayes
2,Brent Barry
3,Craig Smith
4,Stromile Swift


In [288]:
home_cols = ['home_0', 'home_1', 'home_2', 'home_3', 'home_4']
away_cols = ['away_0', 'away_1', 'away_2', 'away_3', 'away_4']

In [289]:
for col in home_cols + away_cols:
    nba_test_df[col] = nba_test_df[col].map(names_dict)

nba_test_df['home_team'] = nba_test_df['home_team'].map(teams_dict) # map team id string to idx
nba_test_df['away_team'] = nba_test_df['away_team'].map(teams_dict) # map team id string to idx
nba_test_df['season'] = nba_test_df['season'].astype(int).map(seasons_dict) # map season int to idx

nba_test_df.tail()

,season,home_team,away_team,starting_min,home_0,home_1,home_2,home_3,home_4,away_0,away_1,away_2,away_3,away_4
995,9,2,5,20,NaN,517.0,988.0,1056.0,1068.0,14,196,594,715,784
996,9,32,5,21,255.0,258.0,NaN,629.0,666.0,358,493,594,715,784
997,9,7,9,13,246.0,NaN,294.0,416.0,857.0,78,600,858,962,999
998,9,22,25,24,62.0,339.0,NaN,561.0,901.0,132,351,474,789,1045
999,9,2,23,20,126.0,297.0,517.0,988.0,NaN,79,143,549,624,878


In [290]:
home_players = []
away_players = []

for index, row in nba_test_df.iterrows():
  current_home_players = []
  current_away_players = []
  for col in home_cols:
    if not pd.isna(row[col]):
      current_home_players.append(int(row[col]))
  for col in away_cols:
    if not pd.isna(row[col]):
      current_away_players.append(int(row[col]))
  home_players.append(current_home_players)
  away_players.append(current_away_players)

nba_test_df['home_players'] = home_players
nba_test_df['away_players'] = away_players

nba_test_df = nba_test_df.drop(columns=home_cols)
nba_test_df = nba_test_df.drop(columns=away_cols)

nba_test_df.tail()

,season,home_team,away_team,starting_min,home_players,away_players
995,9,2,5,20,"[517, 988, 1056, 1068]","[14, 196, 594, 715, 784]"
996,9,32,5,21,"[255, 258, 629, 666]","[358, 493, 594, 715, 784]"
997,9,7,9,13,"[246, 294, 416, 857]","[78, 600, 858, 962, 999]"
998,9,22,25,24,"[62, 339, 561, 901]","[132, 351, 474, 789, 1045]"
999,9,2,23,20,"[126, 297, 517, 988]","[79, 143, 549, 624, 878]"


In [291]:
class BasketballModelMLP(nn.Module):
    def __init__(self, embedding_dim, hidden_dim):
        super(BasketballModelMLP, self).__init__()

        self.player_embedding = nn.Embedding(len(names), embedding_dim)
        self.season_embedding = nn.Embedding(len(seasons), embedding_dim)
        self.team_embedding = nn.Embedding(len(teams), embedding_dim)

        self.home_players_fc = nn.Linear(embedding_dim * 4, hidden_dim)
        self.away_players_fc = nn.Linear(embedding_dim * 5, hidden_dim)
        self.teams_fc = nn.Linear(embedding_dim * 2, hidden_dim)
        self.context_fc = nn.Linear(embedding_dim + 1, hidden_dim)

        self.combined_fc1 = nn.Linear(hidden_dim * 4, hidden_dim * 2)
        self.combined_fc2 = nn.Linear(hidden_dim * 2, hidden_dim)
        self.output_fc = nn.Linear(hidden_dim, len(names))

        self.dropout = nn.Dropout(0.25)

    def forward(self, season_idx, home_team_idx, away_team_idx, starting_min, home_players, away_players):
        batch_size = season_idx.size(0)

        season_idx = season_idx.long()
        home_team_idx = home_team_idx.long()
        away_team_idx = away_team_idx.long()
        home_players = home_players.long()
        away_players = away_players.long()

        # 1. Embeddings
        season_emb = self.season_embedding(season_idx)
        home_team_emb = self.team_embedding(home_team_idx)
        away_team_emb = self.team_embedding(away_team_idx)

        home_player_embeddings = self.player_embedding(home_players)
        home_player_embeddings = home_player_embeddings.view(batch_size, -1)

        away_player_embeddings = self.player_embedding(away_players)
        away_player_embeddings = away_player_embeddings.view(batch_size, -1)

        # 2. Features
        home_player_features = F.relu(self.home_players_fc(home_player_embeddings))
        away_player_features = F.relu(self.away_players_fc(away_player_embeddings))

        context_features = torch.cat([
            season_emb,
            starting_min.unsqueeze(1)
        ], dim=1)
        context_features = F.relu(self.context_fc(context_features))

        teams_features = torch.cat([
            home_team_emb,
            away_team_emb
        ], dim=1)
        teams_features = F.relu(self.teams_fc(teams_features))

        # 3. Combined Features
        combined_features = torch.cat([
            context_features,
            teams_features,
            home_player_features,
            away_player_features
        ], dim=1)

        combined_features = F.relu(self.combined_fc1(combined_features))
        combined_features = self.dropout(combined_features)
        combined_features = F.relu(self.combined_fc2(combined_features))
        combined_features = self.dropout(combined_features)

        logits = self.output_fc(combined_features)

        return logits

In [292]:
def predict_removed_player(model, season_idx, home_team_idx, away_team_idx, starting_min, home_players, away_players, device, names):
    model.eval()

    season_idx = torch.tensor([season_idx], dtype=torch.int32).to(device)
    home_team_idx = torch.tensor([home_team_idx], dtype=torch.int32).to(device)
    away_team_idx = torch.tensor([away_team_idx], dtype=torch.int32).to(device)
    home_players = torch.tensor([home_players], dtype=torch.int32).to(device)
    away_players = torch.tensor([away_players], dtype=torch.int32).to(device)
    starting_min = torch.tensor([starting_min], dtype=torch.float32).to(device)

    with torch.no_grad():
        logits = model(season_idx, home_team_idx, away_team_idx, starting_min, home_players, away_players)
        probs = F.softmax(logits, dim=1)

        confidence, predicted_idx = torch.max(probs, 1)

        predicted_idx_value = predicted_idx.item()
        predicted_player = names[predicted_idx_value] if predicted_idx_value < len(names) else f"Player_{predicted_idx_value}"

        top3_values, top3_indices = torch.topk(probs, 3, dim=1)
        top3_players = []
        for idx, prob in zip(top3_indices[0], top3_values[0]):
            idx_value = idx.item()
            player_name = names[idx_value] if idx_value < len(names) else f"Player_{idx_value}"
            top3_players.append((player_name, prob.item() * 100))

        all_probs = probs[0].cpu().numpy()

    return predicted_player, confidence.item() * 100, top3_players, all_probs

In [293]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [294]:
EMBEDDING_DIM = 64
HIDDEN_DIM = 128

In [295]:
model_path = 'basketball_player_removal_mlp_model_3.pt'
print(f"Loading model from {model_path}")

model = BasketballModelMLP(EMBEDDING_DIM, HIDDEN_DIM)
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()
print("Model loaded successfully")

Loading model from basketball_player_removal_mlp_model_3.pt
Model loaded successfully


In [296]:
predictions = []
confidences = []
top3_predictions = []
true_labels = []
true_player_ranks = []
true_player_probs = []
home_players_names = []
away_players_names = []
seasons_list = []
starting_mins = []
home_teams = []
away_teams = []

print(f"Running predictions on {len(nba_test_df)} test samples...")
for idx, row in tqdm(nba_test_df.iterrows(), total=len(nba_test_df)):
    season_idx = row['season']
    home_team_idx = row['home_team']
    away_team_idx = row['away_team']
    starting_min = row['starting_min']

    home_players_list = row['home_players']
    away_players_list = row['away_players']

    home_players_padded = home_players_list + [0] * (4 - len(home_players_list))
    away_players_padded = away_players_list + [0] * (5 - len(away_players_list))

    predicted_player, confidence, top3, all_probs = predict_removed_player(
        model, season_idx, home_team_idx, away_team_idx, starting_min,
        home_players_padded, away_players_padded, device, names
    )

    predictions.append(predicted_player)
    confidences.append(confidence)
    top3_predictions.append(top3)

    home_players_name = [names[player_idx] for player_idx in home_players_list]
    away_players_name = [names[player_idx] for player_idx in away_players_list]

    home_players_names.append(home_players_name)
    away_players_names.append(away_players_name)
    seasons_list.append(seasons[season_idx])
    starting_mins.append(starting_min)
    home_teams.append(teams[home_team_idx])
    away_teams.append(teams[away_team_idx])

    if idx < len(nba_test_labels_df):
        true_player = nba_test_labels_df.iloc[idx]['removed_value']
        true_labels.append(true_player)

        if true_player in names:
            true_player_idx = names.index(true_player)
            true_player_prob = all_probs[true_player_idx] * 100

            sorted_probs = np.sort(all_probs)[::-1]
            true_rank = np.where(sorted_probs == all_probs[true_player_idx])[0][0] + 1

            true_player_ranks.append(true_rank)
            true_player_probs.append(true_player_prob)
        else:
            true_player_ranks.append(None)
            true_player_probs.append(None)

results_df = pd.DataFrame({
    'season': seasons_list,
    'home_team': home_teams,
    'away_team': away_teams,
    'starting_min': starting_mins,
    'home_players': home_players_names,
    'away_players': away_players_names,
    'predicted_player': predictions,
    'confidence': confidences,
})

if true_labels:
    results_df['true_player'] = true_labels
    results_df['true_player_rank'] = true_player_ranks
    results_df['true_player_probability'] = true_player_probs

    results_df['correct'] = results_df['predicted_player'] == results_df['true_player']
    accuracy = results_df['correct'].mean() * 100

    for k in [1, 3, 5, 10]:
        in_top_k = [rank <= k if rank is not None else False for rank in true_player_ranks]
        top_k_accuracy = sum(in_top_k) / len(in_top_k) * 100
        print(f"Top-{k} accuracy: {top_k_accuracy:.2f}%")

    valid_ranks = [rank for rank in true_player_ranks if rank is not None]
    mean_rank = sum(valid_ranks) / len(valid_ranks) if valid_ranks else None
    median_rank = sorted(valid_ranks)[len(valid_ranks)//2] if valid_ranks else None

    print(f"Overall accuracy: {accuracy:.2f}%")
    print(f"Mean rank of true player: {mean_rank:.2f}")
    print(f"Median rank of true player: {median_rank}")

print("\nSample predictions:")
print(results_df.head(10))

results_df.to_csv('nba_predictions_results.csv', index=False)
print("Predictions saved to 'nba_predictions_results.csv'")

Running predictions on 1000 test samples...


100%|██████████| 1000/1000 [00:01<00:00, 824.95it/s]


Top-1 accuracy: 28.90%
Top-3 accuracy: 53.90%
Top-5 accuracy: 65.30%
Top-10 accuracy: 77.00%
Overall accuracy: 28.90%
Mean rank of true player: 28.15
Median rank of true player: 3

Sample predictions:
   season home_team away_team  starting_min  \
0    2007       IND       BOS            18   
1    2007       HOU       DAL            16   
2    2007       SAS       POR            39   
3    2007       MIN       BOS            21   
4    2007       MEM       LAL            19   
5    2007       MIL       CLE             8   
6    2007       MIA       GSW            11   
7    2007       CLE       SAC            37   
8    2007       GSW       WAS            35   
9    2007       DEN       TOR            10   

                                        home_players  \
0  [Danny Granger, Darrell Armstrong, Keith McLeo...   
1  [Bonzi Wells, Juwan Howard, Luther Head, Tracy...   
2  [Beno Udrih, Bruce Bowen, Matt Bonner, Tim Dun...   
3  [Kevin Garnett, Randy Foye, Ricky Davis, Trent...   
4

In [297]:
season_std_devs = results_df.groupby('season')['true_player_rank'].std()
print("Standard Deviation of True Removed Player Rank by Season:")
print(season_std_devs)

Standard Deviation of True Removed Player Rank by Season:
season
2007     43.070724
2008     30.179262
2009      9.848222
2010     28.969570
2011     38.944367
2012     83.959015
2013     47.577317
2014     18.266874
2015     32.809304
2016    216.385154
Name: true_player_rank, dtype: float64
